In [3]:
# =====================================================
# Predicting F1 Pit Stops | Playground Series S6E5
# Target: High ROC-AUC (0.94+)
# =====================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

import catboost as cb
import xgboost as xgb
import lightgbm as lgb

from tqdm import tqdm
import gc

In [4]:
# Load Data
print("Loading datasets...")
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/test.csv')
submission = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Target distribution:\n{train['PitNextLap'].value_counts(normalize=True)}")

# Save test id
test_id = test['id'] if 'id' in test.columns else None

Loading datasets...
Train shape: (439140, 16)
Test shape: (188165, 15)
Target distribution:
PitNextLap
0.0    0.801018
1.0    0.198982
Name: proportion, dtype: float64


In [5]:
print("Train Columns:")
print(train.columns.tolist())

print("\nTest Columns:")
print(test.columns.tolist())

Train Columns:
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'PitNextLap']

Test Columns:
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']


In [6]:
def feature_engineering(df):
    df = df.copy()
    group_cols = ['Driver', 'Race']   # Most important grouping
    
    # === Tyre Features ===
    if 'TyreLife' in df.columns:
        df['TyreLife_sq'] = df['TyreLife'] ** 2
        df['TyreLife_log'] = np.log1p(df['TyreLife'])
        df['TyreLife_sqrt'] = np.sqrt(df['TyreLife'])
    
    # === Rolling Features ===
    rolling_features = ['Speed', 'LapTime', 'Position', 'TyreLife']
    for col in rolling_features:
        if col in df.columns:
            df[f'{col}_roll_mean_5'] = df.groupby(group_cols)[col].transform(
                lambda x: x.rolling(5, min_periods=1).mean())
            df[f'{col}_roll_std_5'] = df.groupby(group_cols)[col].transform(
                lambda x: x.rolling(5, min_periods=1).std())
            df[f'{col}_roll_mean_10'] = df.groupby(group_cols)[col].transform(
                lambda x: x.rolling(10, min_periods=1).mean())
    
    # === Lag Features ===
    lag_features = ['Speed', 'Position', 'LapTime', 'TyreLife']
    for col in lag_features:
        if col in df.columns:
            df[f'{col}_lag1'] = df.groupby(group_cols)[col].shift(1)
            df[f'{col}_delta1'] = df[col] - df[f'{col}_lag1']
    
    # === Race Context ===
    if 'Lap' in df.columns:
        df['Lap_Pct'] = df.groupby('Race')['Lap'].transform(lambda x: x / x.max())
    
    if 'Position' in df.columns:
        df['Position_Change'] = df.groupby(group_cols)['Position'].diff()
        df['Is_Leader'] = (df['Position'] == 1).astype(int)
    
    # === Interactions ===
    if 'TyreLife' in df.columns and 'Speed' in df.columns:
        df['TyreLife_x_Speed'] = df['TyreLife'] * df['Speed']
    
    if 'Lap' in df.columns and 'Position' in df.columns:
        df['Lap_x_Position'] = df['Lap'] * df['Position']
    
    # Fill missing values
    df = df.fillna(-999)
    return df

print("Feature Engineering Function Ready!")

Feature Engineering Function Ready!


In [7]:
print("Applying Feature Engineering...")
train = feature_engineering(train)
test = feature_engineering(test)

print(f"New Train shape: {train.shape}")
print(f"New Test shape: {test.shape}")

Applying Feature Engineering...
New Train shape: (439140, 30)
New Test shape: (188165, 29)


In [8]:
# Identify categorical columns
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns:", cat_cols)

# Label Encoding
for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

print("Encoding Completed!")

Categorical columns: ['Driver', 'Compound', 'Race']
Encoding Completed!


In [9]:
target = 'PitNextLap'

X = train.drop([target, 'id'], axis=1, errors='ignore')
y = train[target]
X_test = test.drop(['id'], axis=1, errors='ignore')

print(f"Final Features: {X.shape[1]}")
print(X.columns.tolist())

Final Features: 28
['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'TyreLife_sq', 'TyreLife_log', 'TyreLife_sqrt', 'Position_roll_mean_5', 'Position_roll_std_5', 'Position_roll_mean_10', 'TyreLife_roll_mean_5', 'TyreLife_roll_std_5', 'TyreLife_roll_mean_10', 'Position_lag1', 'Position_delta1', 'TyreLife_lag1', 'TyreLife_delta1', 'Is_Leader']


In [10]:
params = {
    'iterations': 1500,
    'learning_rate': 0.07,
    'depth': 8,
    'l2_leaf_reg': 4,
    'random_seed': 42,
    'eval_metric': 'AUC',
    'verbose': 200,
    'early_stopping_rounds': 100,
    'task_type': 'CPU'
}

model = cb.CatBoostClassifier(**params)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\nTraining Fold {fold+1}/5")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], use_best_model=True)
    
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / 5

print(f"\n✅ OOF ROC-AUC Score: {roc_auc_score(y, oof_preds):.5f}")


Training Fold 1/5
0:	test: 0.9018879	best: 0.9018879 (0)	total: 152ms	remaining: 3m 47s
200:	test: 0.9417400	best: 0.9417400 (200)	total: 14.8s	remaining: 1m 35s
400:	test: 0.9451421	best: 0.9451421 (400)	total: 29.3s	remaining: 1m 20s
600:	test: 0.9461962	best: 0.9461962 (600)	total: 43.7s	remaining: 1m 5s
800:	test: 0.9467728	best: 0.9467728 (800)	total: 58.1s	remaining: 50.7s
1000:	test: 0.9471367	best: 0.9471382 (999)	total: 1m 12s	remaining: 36.1s
1200:	test: 0.9473563	best: 0.9473624 (1192)	total: 1m 26s	remaining: 21.6s
1400:	test: 0.9474392	best: 0.9474444 (1369)	total: 1m 40s	remaining: 7.13s
1499:	test: 0.9474490	best: 0.9474490 (1499)	total: 1m 47s	remaining: 0us

bestTest = 0.9474489676
bestIteration = 1499


Training Fold 2/5
0:	test: 0.8997569	best: 0.8997569 (0)	total: 89.7ms	remaining: 2m 14s
200:	test: 0.9399509	best: 0.9399509 (200)	total: 14.8s	remaining: 1m 35s
400:	test: 0.9436255	best: 0.9436255 (400)	total: 29.5s	remaining: 1m 20s
600:	test: 0.9447903	best: 0.94

In [11]:
submission['PitNextLap'] = test_preds
submission.to_csv('submission_f1_pitstop.csv', index=False)
print("Submission file saved!")
submission.head()

Submission file saved!


,id,PitNextLap
0,439140,0.009059
1,439141,0.005077
2,439142,0.006921
3,439143,0.077559
4,439144,0.786424
